In [ ]:
!pip install ipywidgets --quiet

import re, random
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widGets
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import joblib
import matplotlib.pyplot as plt

random.seed(42)
print("Ready ✅")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.8 MB/s eta 0:00:00
Ready ✅


In [ ]:

seed = {
    "depression": [
        "I feel empty and lost", "Nothing seems worth it anymore", "I can't find joy in anything",
        "Life feels meaningless", "I feel hopeless", "I feel like a burden", "I'm tired of pretending",
        "I don't want to get out of bed", "I keep crying without reason", "I feel numb most days"
    ],
    "anxiety": [
        "I can't stop worrying", "My heart races and I panic", "I feel nervous about everything",
        "I'm afraid of making mistakes", "I overthink all the time", "I feel on edge",
        "I dread tomorrow", "I'm scared about the future", "My thoughts spiral", "I fear losing control"
    ],
    "frustration": [
        "I'm so frustrated with myself", "Everything is annoying today", "I keep failing and it hurts",
        "I'm fed up with this situation", "Nothing goes my way", "I feel angry and stuck",
        "I can’t stand repeating the same errors", "This is getting on my nerves",
        "I feel irritated at everyone", "Why won't things just work?"
    ],
    "restless": [
        "I can't relax at all", "I keep pacing around the room", "I feel jittery and unsettled",
        "My mind won’t slow down", "I can’t sit still", "I feel fidgety",
        "I keep tossing and turning all night", "I feel wired but tired",
        "I can’t focus on one thing", "I feel constantly unsettled"
    ],
    "positive": [
        "I feel calm and grateful", "I had a wonderful day", "Everything is falling into place",
        "I'm proud of my progress", "I feel hopeful and strong", "I'm happy with my life",
        "I enjoyed time with family", "I’m excited for tomorrow", "I feel confident",
        "I’m at peace with where I am"
    ]
}


templates = [
    "Today I {verb} {feeling}.",
    "Lately, I {verb} {feeling}.",
    "Sometimes I {verb} {feeling}, and it’s hard.",
    "I just {verb} {feeling}.",
    "Right now, I {verb} {feeling}.",
]

verbalizers = {
    "depression": ["feel", "am feeling", "have been feeling"],
    "anxiety": ["feel", "am feeling", "keep feeling"],
    "frustration": ["feel", "am feeling", "keep getting"],
    "restless": ["feel", "am feeling", "keep feeling"],
    "positive": ["feel", "am feeling", "have been feeling"],
}

synonyms = {
    "depression": ["empty", "lost", "down", "hopeless", "numb", "low", "meaningless"],
    "anxiety": ["anxious", "worried", "uneasy", "panicky", "nervous", "on edge"],
    "frustration": ["frustrated", "irritated", "annoyed", "fed up", "angry", "stuck"],
    "restless": ["restless", "jittery", "unsettled", "fidgety", "wired", "unable to relax"],
    "positive": ["grateful", "hopeful", "calm", "happy", "content", "optimistic", "peaceful"]
}

def cleanSpace(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def generateClassSamples(label, n_target=24):
    base_li = seed[label][:]

    out_li = base_li[:]

    while len(out_li) < n_target:
        tmpl = random.choice(templates)
        verb = random.choice(verbalizers[label])
        feel = random.choice(synonyms[label])
        sent = tmpl.format(verb=verb, feeling=feel)

        extra = [
            "",
            "and I don't know what to do",
            "but I’m trying to cope",
            "and I wish it would stop",
            "though I’m managing better today",
            "and it keeps coming back",
            "but I’m taking it one step at a time"
        ]
        sent = cleanSpace(f"{sent} {random.choice(extra)}.")
        out_li.append(sent)
    return out_li[:n_target]


perClass = 24
allTexts, allLabels = [], []

for lbl in ["depression", "anxiety", "frustration", "restless", "positive"]:
    samples = generateClassSamples(lbl, n_target=perClass)
    allTexts.extend(samples)
    allLabels.extend([lbl] * len(samples))


a = list(zip(allTexts, allLabels))
random.shuffle(a)
all_texts, all_labels = zip(*a)

df = pd.DataFrame({"text": allTexts, "label": allLabels})
df.to_csv("dataMulticlass.csv", index=False)

print(df['label'].value_counts())
df.head(8)



label
depression     24
anxiety        24
frustration    24
restless       24
positive       24
Name: count, dtype: int64


,text,label
0,I feel empty and lost,depression
1,Nothing seems worth it anymore,depression
2,I can't find joy in anything,depression
3,Life feels meaningless,depression
4,I feel hopeless,depression
5,I feel like a burden,depression
6,I'm tired of pretending,depression
7,I don't want to get out of bed,depression


In [ ]:

df = pd.read_csv("dataMulticlass.csv")


XTrain, XTest, yTrain, yTest = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)


model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ("clf", MultinomialNB())
])

model.fit(XTrain, yTrain)


yPred = model.predict(XTest)
print(classification_report(yTest, yPred, digits=3))

joblib.dump(model, "Mental_Health_Multiclass.pkl")

print("Model trained and saved ✅")


              precision    recall  f1-score   support

     anxiety      0.300     0.600     0.400         5
  depression      0.000     0.000     0.000         4
 frustration      1.000     0.600     0.750         5
    positive      0.429     0.600     0.500         5
    restless      0.000     0.000     0.000         5

    accuracy                          0.375        24
   macro avg      0.346     0.360     0.330        24
weighted avg      0.360     0.375     0.344        24

Model trained and saved ✅


In [ ]:
model = joblib.load("Mental_Health_Multiclass.pkl")

def predictPercentages(text: str):
    proba = model.predict_proba([text])[0]
    classes = list(model.classes_)
    return {cls: round(float(p)*100, 2) for cls, p in zip(classes, proba)}

def topLabel(probs: dict):
    return max(probs.items(), key=lambda kv: kv[1])

def colorFor(label: str):
    return {
        "depression": "#cc0000",
        "anxiety": "#e67e22",
        "frustration": "#c0392b",
        "restless": "#8e44ad",
        "positive": "#2e7d32",
    }.get(label, "#333333")

print("Ready to predict ✅")


Ready to predict ✅


In [ ]:

inp = widGets.Textarea(
    value='',
    place_holder="Please type how you're feeling __ (e.g., 'I'm not feeling good and I feel frustrated about everything today')",
    des_cription='Input:',
    layout=widGets.Layout(width='100%', height='100px')
)

btn = widGets.Button(description='Analyze', button_style='success')
out = widGets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})
hist = widGets.Output(layout={'border': '1px dashed #bbb', 'padding': '8px'})

historyItems = []

def drawBar(probs: dict):

    items = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)
    labels = [k for k,_ in items]
    values = [v for _,v in items]
    colors = [colorFor(k) for k in labels]
    plt.figure(figsize=(6,3))
    plt.bar(labels, values, color=colors)
    plt.ylim(0, 100)
    plt.ylabel("Percentage")
    plt.title("Emotion probabilities")
    plt.tight_layout()
    plt.show()

def renderHistory():
    with hist:
        clear_output()
        display(HTML("<h4 style='margin:0'>📜 History (last 10)</h4><hr/>"))
        for item in historyItems[-10:][::-1]:
            label, pct = item['top']
            color = colorFor(label)
            display(HTML(
                f"<div style='margin-bottom:8px'>"
                f"<div><b>Input:</b> {item['text']}</div>"
                f"<div><b>Top:</b> <span style='color:{color}'>{label.title()} — {pct:.1f}%</span></div>"
                f"</div><hr/>"
            ))

def onClick(_):
    with out:
        clear_output()
        text = inp.value.strip()
        if not text:
            display(HTML("<span style='color:#777'>Please enter some text.</span>"))
            return
        probs = predictPercentages(text)
        label, pct = topLabel(probs)
        # Emoji and headline color
        emoji = {"positive":"🙂","anxiety":"😰","frustration":"😠","depression":"😔","restless":"😵"}.get(label, "🧠")
        color = colorFor(label)
        display(HTML(
            f"<h3 style='margin:4px 0;color:{color}'>{emoji} {label.title()} — {pct:.1f}%</h3>"
        ))

        rows = "".join(
            f"<tr><td style='padding:4px 8px'>{k.title()}</td>"
            f"<td style='padding:4px 8px;text-align:right'>{v:.2f}%</td></tr>"
            for k,v in sorted(probs.items(), key=lambda kv: kv[1], reverse=True)
        )
        display(HTML(
            "<table style='border-collapse:collapse'>"
            "<thead><tr><th style='text-align:left;padding:4px 8px'>Label</th>"
            "<th style='text-align:right;padding:4px 8px'>Probability</th></tr></thead>"
            f"<tbody>{rows}</tbody></table>"
        ))

        drawBar(probs)

        historyItems.append({"text": text, "probs": probs, "top": (label, pct)})
        renderHistory()

btn.on_click(onClick)

display(HTML("<h2 style='margin-bottom:6px'>🧠 Mental Health Text Classifier</h2>"
             "<p style='margin-top:0;color:#555'>This demo estimates the likelihood of "
             "<b>depression</b>, <b>anxiety</b>, <b>frustration</b>, <b>restless</b>, and <b>positive</b> "
             "from a short text</p>"))
UI = widGets.VBox([inp, btn, out, hist])
display(UI)


